# 沪深300策略回测可视化报告

本 Notebook 基于 `src/` 中的回测模块进行交互式可视化分析，包括：
1. 行情数据概览
2. K线蜡烛图 + 均线 + 买卖信号标记
3. 策略净值与回撤分析

In [ ]:
import sys
sys.path.append("../src")

import matplotlib.pyplot as plt
import mplfinance as mpf

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

from data_loader import load_index_data
from strategy import compute_signals
from backtest import run_backtest
from metrics import drawdown_series

## 1. 加载沪深300指数数据

In [ ]:
df_raw = load_index_data(start_date="2006-01-01")
print(f"数据区间：{df_raw.index.min().date()} ~ {df_raw.index.max().date()}，共 {len(df_raw)} 个交易日")
df_raw.tail()

## 2. K线蜡烛图 + 买卖信号标记

取最近 3 年数据，用 MA(8/30) 策略演示信号生成效果：

In [ ]:
plot_df = df_raw.iloc[-756:].copy()
plot_df = compute_signals(plot_df, "ma", p1=8, p2=30)

ap = [
    mpf.make_addplot(plot_df["ma_s"], color="red", label="短均线MA8"),
    mpf.make_addplot(plot_df["ma_l"], color="blue", label="长均线MA30"),
    mpf.make_addplot(plot_df["trend_ma"], color="green", linestyle="--", label="MA250趋势线"),
]

fig, ax = mpf.plot(
    plot_df, type="candle", addplot=ap, figsize=(14, 7),
    title="沪深300蜡烛K线 + 买卖信号标记", ylabel="价格",
    returnfig=True, volume=False,
)

buy_pts = plot_df[plot_df["signal"] == 1]
sell_pts = plot_df[plot_df["signal"] == -1]
ax[0].scatter(buy_pts.index, buy_pts["Low"] * 0.98, marker="^", color="#00bb00", s=60, zorder=10, label="买入")
ax[0].scatter(sell_pts.index, sell_pts["High"] * 1.02, marker="v", color="#ff2222", s=60, zorder=10, label="卖出")
ax[0].legend(loc="upper left")
plt.show()

## 3. 单次回测：净值 vs 买入持有

In [ ]:
res, _ = run_backtest(plot_df, "ma", p1=8, p2=30)
if res:
    valid = res["df_valid"]
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax1.plot(valid.index, valid["cum_strategy"], label="MA策略净值", color="crimson")
    ax1.plot(valid.index, valid["cum_bench"], label="买入持有", color="gray", ls="--")
    ax1.set_title("策略净值 vs 买入持有")
    ax1.legend(); ax1.grid(alpha=0.3)
    dd = drawdown_series(valid["cum_strategy"])
    ax2.fill_between(dd.index, dd, 0, color="crimson", alpha=0.4)
    ax2.set_title("策略回撤"); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f"年化收益：{res['ann_ret']:.2%} | 夏普：{res['sharpe']:.2f} | 最大回撤：{res['max_dd']:.2%} | 交易次数：{res['trade_cnt']}")

## 4. 小结

完整的滚动窗口样本外回测请运行 `src/backtest.py`，绩效汇总输出至 `results/backtest_summary.csv`。